# تشغيل كاشف الورشة من GitHub على Colab

الدفتر ينزّل النموذج المدرّب من GitHub، يتيح لك رفع صورة أو أكثر، ثم يعرض الصور مع المربعات ويحمل النتائج. لتشغيل أسرع، اختَر **Runtime → Change runtime type → T4 GPU** قبل **Run all**. يعمل على CPU أيضاً.


In [ ]:
from pathlib import Path
import subprocess, sys, shutil, zipfile
import torch
from IPython.display import display, Image as DisplayImage
from google.colab import files

repo = Path('/content/WoodWorkShop')
if not repo.exists():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/omar-salem23/WoodWorkShop.git',str(repo)])
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
print('Runtime:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Model:', repo/'models/best.pt')


## ارفع الصور

اختر الصور من جهازك عند ظهور نافذة الرفع. الأنواع الشائعة مثل JPG وPNG مدعومة.


In [ ]:
from google.colab import files
upload_dir=Path('/content/workshop_input'); upload_dir.mkdir(exist_ok=True)
uploaded=files.upload()
inputs=[]
for name,content in uploaded.items():
    path=upload_dir/Path(name).name
    path.write_bytes(content)
    inputs.append(path)
if not inputs:
    raise ValueError('ارفع صورة واحدة على الأقل')
print('Images:',len(inputs))


## شغّل النموذج واعرض النتائج

الافتراضي يختار GPU إذا كان متاحاً. غيّر `confidence` إذا أردت إخفاء التوقعات الأقل ثقة؛ مثلاً 0.40 بدلاً من 0.25.


In [ ]:
from ultralytics import YOLO

model=YOLO(str(repo/'models/best.pt'))
confidence=0.25
device='0' if torch.cuda.is_available() else 'cpu'
out=repo/'runs/colab_predictions'
results=model.predict(source=[str(p) for p in inputs],conf=confidence,device=device,
    save=True,project=str(out),name='results',exist_ok=True,verbose=False)
for result in results:
    counts={}
    for box in result.boxes:
        label=model.names[int(box.cls.item())]
        counts[label]=counts.get(label,0)+1
    print(Path(result.path).name, counts or 'بدون اكتشافات')
    display(DisplayImage(data=result.plot()[:,:,::-1].copy()))
print('Annotated outputs:',out/'results')


## نزّل الصور المعلّمة

تُجمع النتائج في ملف ZIP وينزّله المتصفح إلى جهازك.


In [ ]:
archive=shutil.make_archive('/content/workshop_predictions','zip',out/'results')
files.download(archive)
